In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadnafian/kdef-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'kdef-dataset' dataset.
Path to dataset files: /kaggle/input/kdef-dataset


In [ ]:
import os

# Gather unique directory paths containing images
unique_dirs = set()
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            # Get the path relative to the directory containing 'kdef-database'
            base_dir = os.path.dirname(os.path.dirname(os.path.dirname(path)))
            relative_dir = os.path.relpath(root, base_dir)
            unique_dirs.add(relative_dir)

# Print the paths (without filenames)
for d in sorted(list(unique_dirs)):
    print(f"{d}/")

kaggle/input/kdef-dataset/KDEF/AF01/
kaggle/input/kdef-dataset/KDEF/AF02/
kaggle/input/kdef-dataset/KDEF/AF03/
kaggle/input/kdef-dataset/KDEF/AF04/
kaggle/input/kdef-dataset/KDEF/AF05/
kaggle/input/kdef-dataset/KDEF/AF06/
kaggle/input/kdef-dataset/KDEF/AF07/
kaggle/input/kdef-dataset/KDEF/AF08/
kaggle/input/kdef-dataset/KDEF/AF09/
kaggle/input/kdef-dataset/KDEF/AF10/
kaggle/input/kdef-dataset/KDEF/AF11/
kaggle/input/kdef-dataset/KDEF/AF12/
kaggle/input/kdef-dataset/KDEF/AF13/
kaggle/input/kdef-dataset/KDEF/AF14/
kaggle/input/kdef-dataset/KDEF/AF15/
kaggle/input/kdef-dataset/KDEF/AF16/
kaggle/input/kdef-dataset/KDEF/AF17/
kaggle/input/kdef-dataset/KDEF/AF18/
kaggle/input/kdef-dataset/KDEF/AF19/
kaggle/input/kdef-dataset/KDEF/AF20/
kaggle/input/kdef-dataset/KDEF/AF21/
kaggle/input/kdef-dataset/KDEF/AF22/
kaggle/input/kdef-dataset/KDEF/AF23/
kaggle/input/kdef-dataset/KDEF/AF24/
kaggle/input/kdef-dataset/KDEF/AF25/
kaggle/input/kdef-dataset/KDEF/AF26/
kaggle/input/kdef-dataset/KDEF/AF27/
k

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

# 1. Gather all image files from the dataset path
image_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            image_files.append(os.path.join(root, file))

print(f"Found {len(image_files)} images in the dataset.")

Found 4900 images in the dataset.


In [ ]:
# 2. Define the preprocessing pipeline
# Standard ImageNet normalization values are often used when resizing to 224x224
preprocess_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),        # Resize to 224x224
    transforms.Lambda(lambda x: x.convert("RGB")), # Convert to RGB (3 channels)
    transforms.ToTensor(),                # Convert to Tensor (0-1 range)
    transforms.Normalize(                 # Normalize
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Apply to a sample image
if image_files:
    sample_image_path = image_files[0]
    original_image = Image.open(sample_image_path)

    # Apply preprocessing
    processed_tensor = preprocess_pipeline(original_image)

    print(f"Original Size: {original_image.size}")
    print(f"Processed Tensor Shape: {processed_tensor.shape}")
    print(f"Tensor Stats - Min: {processed_tensor.min():.3f}, Max: {processed_tensor.max():.3f}, Mean: {processed_tensor.mean():.3f}")
else:
    print("No images found to process.")

Original Size: (562, 762)
Processed Tensor Shape: torch.Size([3, 224, 224])
Tensor Stats - Min: -1.966, Max: 1.393, Mean: -0.572


In [ ]:
import os
import torch
from tqdm import tqdm
from PIL import Image

# Define output directory in Drive
output_dir = '/content/drive/MyDrive/processed_kdef_pt'

print(f"Saving processed tensors to: {output_dir}")

# We will use the 'preprocess_pipeline' defined in the previous cell
successful_saves = 0
dataset_root = path

# Define the mapping for classes based on the filename's emotion code
emotion_map = {
    "AN": "Angry",
    "DI": "Disgust",
    "AF": "Fear",
    "HA": "Happy",
    "NE": "Neutral",
    "SA": "Sad",
    "SU": "Surprise"
}

for img_path in tqdm(image_files, desc="Processing Images"):
    try:
        # Open image
        img = Image.open(img_path)

        # Apply full preprocessing pipeline (returns a torch.Tensor)
        processed_tensor = preprocess_pipeline(img)

        # Extract filename (e.g., 'AM17ANS.JPG')
        original_filename = os.path.basename(img_path)

        # The emotion code is characters 4 and 5 in the filename (e.g., 'AM17ANS.JPG' -> 'AN')
        if len(original_filename) >= 6:
            emotion_code = original_filename[4:6].upper()
        else:
            emotion_code = "UNKNOWN"

        # Get the corresponding emotion folder name
        new_folder_name = emotion_map.get(emotion_code, "Unknown")

        # Change the file extension from image format to .pt and prepend 'kdef_'
        base_name = os.path.splitext(original_filename)[0]
        new_filename = f'kdef_{base_name}.pt'

        # Construct full save path grouped by emotion class
        save_path = os.path.join(output_dir, new_folder_name, new_filename)

        # Ensure the subdirectory exists
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # Save as a PyTorch tensor file
        torch.save(processed_tensor, save_path)
        successful_saves += 1
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

print(f"Successfully saved {successful_saves} tensors to Google Drive, grouped into 7 emotion classes.")

Saving processed tensors to: /content/drive/MyDrive/processed_kdef_pt


Processing Images: 100%|██████████| 4900/4900 [25:29<00:00,  3.20it/s]

Successfully saved 4900 tensors to Google Drive, grouped into 7 emotion classes.


In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Directory where tensors are saved
source_dir = '/content/drive/MyDrive/processed_kdef_pt'
# New directory for the split datasets
split_dir = '/content/drive/MyDrive/processed_kdef_split'

# 1. Gather all file paths, their corresponding labels, and subjects
data = []
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith('.pt'):
            # The label is the name of the folder containing the file
            label = os.path.basename(root)
            filepath = os.path.join(root, file)

            # Extract subject from filename. Example: kdef_BM23ANHL.pt -> BM23
            # Remove 'kdef_' prefix, then take first 4 chars
            base_name = file.replace('kdef_', '').split('.')[0]
            subject = base_name[:4]

            data.append({'filepath': filepath, 'label': label, 'subject': subject})

df = pd.DataFrame(data)

# 2. Perform the 80:20 Split based on SUBJECTS
if not df.empty:
    unique_subjects = df['subject'].unique()
    train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)

    # Filter dataframe based on subject assignments
    train_df = df[df['subject'].isin(train_subjects)]
    test_df = df[df['subject'].isin(test_subjects)]

    print(f"Total files: {len(df)}")
    print(f"Total subjects: {len(unique_subjects)}")
    print(f"Train subjects: {len(train_subjects)} | Train files: {len(train_df)}")
    print(f"Test subjects: {len(test_subjects)} | Test files: {len(test_df)}\n")

    # 3. Helper function to copy files to the new structure
    def copy_files(dataframe, split_name):
        print(f"\nCopying {split_name} files...")
        for _, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
            src_path = row['filepath']
            label = row['label']
            filename = os.path.basename(src_path)

            # Construct target directory (e.g., .../processed_kdef_split/train/Sad)
            target_dir = os.path.join(split_dir, split_name, label)
            os.makedirs(target_dir, exist_ok=True)

            # Copy the file
            target_path = os.path.join(target_dir, filename)
            shutil.copy2(src_path, target_path)

    # 4. Copy the files into train and test directories
    copy_files(train_df, 'train')
    copy_files(test_df, 'test')

    print(f"\nSuccessfully copied files to: {split_dir}")
else:
    print("No .pt files found to process. Please ensure the tensors were generated correctly.")

Total files: 4900
Total subjects: 140
Train subjects: 112 | Train files: 3920
Test subjects: 28 | Test files: 980


Copying train files...


100%|██████████| 3920/3920 [07:06<00:00,  9.19it/s]



Copying test files...


100%|██████████| 980/980 [00:25<00:00, 37.92it/s]


Successfully copied files to: /content/drive/MyDrive/processed_kdef_split


In [ ]:
import os

dir_path = '/content/drive/MyDrive/processed_kdef_pt'

if os.path.exists(dir_path):
    total_files = sum(len(files) for root, dirs, files in os.walk(dir_path))
    print(f"Total files in '{dir_path}': {total_files}")
else:
    print(f"Directory '{dir_path}' does not exist.")

Total files in '/content/drive/MyDrive/processed_kdef_pt': 4900


In [ ]:
import os
import pandas as pd
from collections import Counter

# Path to the directory where we saved the splits
dir_path = '/content/drive/MyDrive/processed_kdef_split'

train_counts = Counter()
test_counts = Counter()

# Count images by split and class directly from the folders
if os.path.exists(dir_path):
    for split_name in ['train', 'test']:
        split_path = os.path.join(dir_path, split_name)
        if os.path.exists(split_path):
            for class_name in os.listdir(split_path):
                class_dir = os.path.join(split_path, class_name)
                if os.path.isdir(class_dir):
                    num_files = len([f for f in os.listdir(class_dir) if f.endswith('.pt')])
                    if split_name == 'train':
                        train_counts[class_name] += num_files
                    elif split_name == 'test':
                        test_counts[class_name] += num_files

# Get all unique classes
all_classes = sorted(list(set(train_counts.keys()).union(set(test_counts.keys()))))

# Create a DataFrame for visualization
df_counts = pd.DataFrame({
    'Class': all_classes,
    'Train Count': [train_counts.get(c, 0) for c in all_classes],
    'Test Count': [test_counts.get(c, 0) for c in all_classes]
})

# Add Total column
df_counts['Total'] = df_counts['Train Count'] + df_counts['Test Count']

# Add a Total row at the bottom
total_row = pd.DataFrame({
    'Class': ['All Classes'],
    'Train Count': [df_counts['Train Count'].sum()],
    'Test Count': [df_counts['Test Count'].sum()],
    'Total': [df_counts['Total'].sum()]
})

df_counts = pd.concat([df_counts, total_row], ignore_index=True)

# Display the results
display(df_counts)


,Class,Train Count,Test Count,Total
0,Angry,560,140,700
1,Disgust,560,140,700
2,Fear,560,140,700
3,Happy,560,140,700
4,Neutral,560,140,700
5,Sad,559,140,699
6,Surprise,560,139,699
7,Unknown,1,1,2
8,All Classes,3920,980,4900


In [ ]:
from google.colab import runtime
runtime.unassign()